In [1]:
import os
from datetime import datetime
from pathlib import Path

from tqdm.auto import tqdm

from scripts import (
    anisotropy,
    config_loader,
    geostat,
    ml,
    postproc,
    preproc_data,
    preproc_grid,
    preproc_ml,
    read,
    visualisation,
    write,
    xval,
)

In [2]:
cfg = config_loader.load_config(Path(os.getcwd()) / "config.yaml")



loading config from c:\WERK\GIT\FRESHEM_3D_interpolation\config.yaml


In [3]:
data_g = read.dataset(cfg["path_preproc_data_gridded"])
pred = read.dataset(cfg["path_preproc_prediction_grid"])

In [4]:
import pandas as pd
import numpy as np
import isatis as isa

In [5]:
### DIT IS BETERE OUTPUT DB CREATOR DAN IN GEOSTAT.KRIGING
def isatis_output_db(ds):
    # Grid metadata direct from ds
    grid = isa.GridGeom(
        origin=[ds["x"].values.min(), ds["y"].values.min(), ds["z"].values.min()],
        cell_size=[ds.attrs["cellsize_x"], ds.attrs["cellsize_y"], ds.attrs["cellsize_z"]],
        nxyz=[ds.sizes["x"], ds.sizes["y"], ds.sizes["z"]],
        ndim=3,
    )

    n_cells = ds.sizes["x"] * ds.sizes["y"] * ds.sizes["z"]
    df_out = pd.DataFrame({"cell_id": np.arange(n_cells, dtype=np.int64)})

    # all data vars to isatis DbPandas
    for var_name, da in ds.data_vars.items():
        da_zyx = da.transpose("z", "y", "x")
        df_out[var_name] = np.asarray(da_zyx.data).reshape(-1, order="F")

    return isa.DbPandas(df_out, grid=grid)


In [6]:
data_g

<xarray.Dataset> Size: 202MB
Dimensions:      (z: 258, y: 136, x: 96)
Coordinates:
  * z            (z) float64 2kB -120.8 -120.2 -119.8 -119.2 ... 6.75 7.25 7.75
  * y            (y) float64 1kB 3.911e+05 3.912e+05 ... 3.978e+05 3.979e+05
  * x            (x) float64 768B 3.942e+04 3.948e+04 ... 4.412e+04 4.418e+04
    spatial_ref  int64 8B ...
Data variables: (12/15)
    P(rho≤1)     (z, y, x) float32 13MB ...
    P(rho≤2)     (z, y, x) float32 13MB ...
    P(rho≤3)     (z, y, x) float32 13MB ...
    P(rho≤5)     (z, y, x) float32 13MB ...
    P(rho≤10)    (z, y, x) float32 13MB ...
    P(rho≤20)    (z, y, x) float32 13MB ...
    ...           ...
    line_no      (z, y, x) float32 13MB ...
    short_dist   (z, y, x) float32 13MB ...
    short_angle  (z, y, x) float32 13MB ...
    long_dist    (z, y, x) float32 13MB ...
    long_angle   (z, y, x) float32 13MB ...
    magnitude    (z, y, x) float32 13MB ...
Attributes:
    cellsize_x:  50
    cellsize_y:  50
    cellsize_z:  0.5

In [7]:
var_angle = "long_angle"
var_short_dist = "short_dist"
df_data_g = data_g.to_dataframe()[[var_angle, var_short_dist]].dropna().reset_index()
df_data_g[var_angle + '_rad'] = np.deg2rad(df_data_g[var_angle])
df_data_g['u'] = np.sin(2 * df_data_g[var_angle + '_rad'])
df_data_g['v'] = np.cos(2 * df_data_g[var_angle + '_rad'])


In [8]:
# Create Isatis input database
input_db = isa.DbPandas(df_data_g)

In [9]:
output_db = isatis_output_db(pred)

In [10]:

# anisotropy angle interpolation 
nh_dist_xy = cfg['aniso_interp_angle_nh_dist_xy']
nh_dist_z = cfg['aniso_interp_angle_nh_dist_z']
nh_n_max = cfg['aniso_interp_angle_nh_n_max']

In [11]:
if 1==2:
    # QuickInterpolation calculator
    runner = isa.QuickInterpol()
    runner.set_input_data(input_db, coords=['x', 'y', 'z'], invars=['u', 'v'])
    runner.set_output_data(output_db, sel='mask')

    # neighbourhood parameters
    neigh = isa.Neigh(ellipsoid_size=[nh_dist_xy, nh_dist_xy, nh_dist_z], max_neigh_per_sector=nh_n_max)

    # Inverse distances
    output_db = runner.inverse_distances(neigh=neigh, power=2)

    # Isatis database to dataframe
    output_df = output_db.df()
    output_df.columns = output_df.columns.str.removesuffix("_inverse_dist")

    # Calculate angle from interpolated u and v
    output_df['theta_rad'] = 0.5 * np.arctan2(output_df['u'], output_df['v'])
    output_df['theta_deg'] = np.rad2deg(output_df['theta_rad']) % 180

    #dummy angle of 0 if no calculated angle within mask 
    output_df.loc[(output_df['mask'] == 1) & (output_df['theta_deg'].isna()), 'theta_deg'] = 0

    ### DIT IS BETERE REASSIGN DAN IN GEOSTAT.KRIGING. Oh nee, houdt geen rekening met al bestaande data

    # Zorg dat de volgorde exact overeenkomt met de oorspronkelijke cell_id-opbouw
    output_df = output_df.sort_values("cell_id")

    # 1D -> 3D in (x, y, z), want zo is cell_id gemaakt
    angle_xyz = output_df["theta_deg"].to_numpy().reshape(
        pred.sizes["x"], pred.sizes["y"], pred.sizes["z"]
    ).astype("float32")

    # Terugzetten naar pred met dimensievolgorde (z, y, x)
    pred = pred.assign(
        long_angle=(("z", "y", "x"), np.transpose(angle_xyz, (2, 1, 0)))
    )

    pred.sel(z=-4.75)["long_angle"].plot()

In [12]:
cellsize_xy = cfg['cellsize_xy']
cellsize_z = cfg['cellsize_z']

# neighbourhood parameters
nh_dist_xy = cellsize_xy + 1
nh_dist_z = cellsize_z + 0.01

# QuickInterpolation calculator
runner = isa.QuickInterpol()
runner.set_input_data(input_db, coords=['x', 'y', 'z'], invars=[var_short_dist])
runner.set_output_data(output_db, sel='mask')

# Nearest neighbour to determine cells to fill
output_db = runner.nearest_neighbor(ellipsoid_size=[nh_dist_xy, nh_dist_xy, nh_dist_z])

# Isatis database to dataframe
output_df = output_db.df()
output_df.columns = output_df.columns.str.removesuffix("_nearest_neigh")
output_df



,cell_id,mask,short_dist
0,0,False,NaN
1,1,False,NaN
2,2,False,NaN
3,3,False,NaN
4,4,False,NaN
...,...,...,...
3368443,3368443,False,NaN
3368444,3368444,False,NaN
3368445,3368445,False,NaN
3368446,3368446,False,NaN


In [14]:
output_db.df().dropna()

,cell_id,mask,short_dist
674903,674903,True,353.553406
674904,674904,True,353.553406
674905,674905,True,353.553406
674906,674906,True,353.553406
674907,674907,True,353.553406
...,...,...,...
2731424,2731424,True,291.547607
2731425,2731425,True,291.547607
2731426,2731426,True,291.547607
2731683,2731683,True,291.547607


In [25]:
output_db.df()[var_short_dist].dropna()

674903     353.553406
674904     353.553406
674905     353.553406
674906     353.553406
674907     353.553406
              ...    
2731424    291.547607
2731425    291.547607
2731426    291.547607
2731683    291.547607
2766513    291.547607
Name: short_dist, Length: 7072, dtype: float32